In [2]:
# ============================================================
# Cell 1: Load, check and time-fix all air quality files
# Covers Step 1 (load + stack), Step 2 (check files), Step 3 (fix time)
# ============================================================
import pandas as pd

# ------------------------------------------------------------
# PART A: The list of all 11 files
# Each row = (city name, country code, file path)
# Pakistan is listed first, then India, so the final table follows this order
# ------------------------------------------------------------
files = [
    # Pakistan
    ("Faisalabad",     "PK", "Dataset/Pakistan/Air Quality/Faisalabad - Air.csv"),
    ("Islamabad",      "PK", "Dataset/Pakistan/Air Quality/Islamabad - Air.csv"),
    ("Lahore",         "PK", "Dataset/Pakistan/Air Quality/Lahore - Air.csv"),
    ("Multan",         "PK", "Dataset/Pakistan/Air Quality/Multan - Air.csv"),
    ("Rahim Yar Khan", "PK", "Dataset/Pakistan/Air Quality/Rahim Yar Khan - Air.csv"),
    ("Rawalpindi",     "PK", "Dataset/Pakistan/Air Quality/Rawalpindi - Air.csv"),
    ("Sialkot",        "PK", "Dataset/Pakistan/Air Quality/Sialkot - Air.csv"),
    # India
    ("Amritsar",  "IN", "Dataset/India/Air Quality/Amritsar - AQ.csv"),
    ("Ferozepur", "IN", "Dataset/India/Air Quality/Ferozepur - AQ.csv"),
    ("Jalandhar", "IN", "Dataset/India/Air Quality/Jalandhar - AQ.csv"),
    ("Ludhiana",  "IN", "Dataset/India/Air Quality/Ludhiana - AQ.xlsx"),
]

# ------------------------------------------------------------
# PART B: Short names for the pollutant columns
# If a raw column name STARTS WITH the left text, we rename it to the right text
# ------------------------------------------------------------
short_names = {
    "pm10": "pm10_ugm3",
    "pm2_5": "pm2_5_ugm3",
    "nitrogen_dioxide": "no2_ugm3",
    "sulphur_dioxide": "so2_ugm3",
    "ozone": "o3_ugm3",
    "carbon_monoxide": "co_ugm3",
    "dust": "dust_ugm3",
}

# ------------------------------------------------------------
# PART C: Load every file, check it, and collect it
# ------------------------------------------------------------
all_tables = []        # each city's table will be put in this list
first_columns = None   # will remember the columns of the first file to compare with

print("=== STEP 2: checks on each raw file ===")

for city, country, path in files:

    # ----- STEP 1: read the file (skip the 3 metadata rows at the top) -----
    if path.endswith(".xlsx"):
        df = pd.read_excel(path, skiprows=3)    # Excel file (Ludhiana)
    else:
        df = pd.read_csv(path, skiprows=3)      # normal CSV file

    # ----- Rename the pollutant columns by NAME to short names -----
    new_columns = []
    for col in df.columns:
        new_name = col                          # start by keeping the old name
        for start_of_name, clean_name in short_names.items():
            if col.startswith(start_of_name):   # does the old name start with this text?
                new_name = clean_name           # if yes, use the short name
        new_columns.append(new_name)
    df.columns = new_columns

    # ----- Make "time" a real date-time, and call it time_utc (it is in GMT) -----
    df["time"] = pd.to_datetime(df["time"])
    df = df.rename(columns={"time": "time_utc"})

    # ----- Sort this city's rows from oldest to newest hour -----
    df = df.sort_values("time_utc").reset_index(drop=True)

    # ----- Add the city and country columns (same value on every row) -----
    df["city"] = city
    df["country"] = country

    # ----- STEP 2: checks for this file -----
    # 1) Same columns as the first file?
    this_columns = sorted(df.columns)
    if first_columns is None:
        first_columns = this_columns            # first file: just remember its columns
        columns_ok = True
    else:
        columns_ok = (this_columns == first_columns)

    # 2) Duplicate timestamps (the same hour appearing twice)
    duplicates = df["time_utc"].duplicated().sum()

    # 3) Gaps: places where the next row is NOT exactly 1 hour after the previous
    gaps = (df["time_utc"].diff().dropna() != pd.Timedelta(hours=1)).sum()

    # 4) Print the results for this file
    print(f"{city:15s} rows={len(df):6d}  first={df['time_utc'].min()}  "
          f"last={df['time_utc'].max()}  duplicates={duplicates}  "
          f"gaps={gaps}  same_columns={columns_ok}")

    # Put this city's table in the list
    all_tables.append(df)

# ------------------------------------------------------------
# PART D: Stack all 11 tables on top of each other
# The order stays as in the files list: Pakistan cities first, then India
# Within each city the rows stay oldest to newest
# ------------------------------------------------------------
aq = pd.concat(all_tables, ignore_index=True)

# ------------------------------------------------------------
# PART E: STEP 3 - fix the time
# ------------------------------------------------------------
# Pakistan and Indian Punjab: the raw time is GMT, so add 5 hours for local time
# (India is actually 5.5 hours ahead, but the plan uses +5 for all cities)
aq["time_local"] = aq["time_utc"] + pd.Timedelta(hours=5)

# Split the local time into a date column and an hour column
aq["date"] = aq["time_local"].dt.normalize()   # the date, with the time set to 00:00
aq["hour"] = aq["time_local"].dt.hour          # the hour of the day, 0 to 23

# Keep only 2022-01-01 to 2023-12-31 (2024 stays untouched, and so do 2025-2026)
rows_before = len(aq)
keep = (aq["date"] >= pd.Timestamp("2022-01-01")) & (aq["date"] <= pd.Timestamp("2023-12-31"))
aq = aq[keep].reset_index(drop=True)
print(f"\nRows before cutting: {rows_before}, after cutting: {len(aq)}")

# Put the columns in a clean order
aq = aq[["city", "country", "time_utc", "time_local", "date", "hour",
         "pm10_ugm3", "pm2_5_ugm3", "no2_ugm3", "so2_ugm3",
         "o3_ugm3", "co_ugm3", "dust_ugm3"]]

# ------------------------------------------------------------
# PART F: Final check after cleaning
# ------------------------------------------------------------
print("\nCity order in the table:", list(aq["city"].unique()))
print("Number of cities:", aq["city"].nunique())
print("Shape:", aq.shape)
print(aq.groupby("city", sort=False)["date"].agg(["min", "max", "count"]))
print(aq.head())

=== STEP 2: checks on each raw file ===
Faisalabad      rows= 41208  first=2022-01-06 00:00:00  last=2026-09-18 23:00:00  duplicates=0  gaps=0  same_columns=True
Islamabad       rows= 41208  first=2022-01-06 00:00:00  last=2026-09-18 23:00:00  duplicates=0  gaps=0  same_columns=True
Lahore          rows= 41208  first=2022-01-06 00:00:00  last=2026-09-18 23:00:00  duplicates=0  gaps=0  same_columns=True
Multan          rows= 41208  first=2022-01-06 00:00:00  last=2026-09-18 23:00:00  duplicates=0  gaps=0  same_columns=True
Rahim Yar Khan  rows= 41208  first=2022-01-06 00:00:00  last=2026-09-18 23:00:00  duplicates=0  gaps=0  same_columns=True
Rawalpindi      rows= 41208  first=2022-01-06 00:00:00  last=2026-09-18 23:00:00  duplicates=0  gaps=0  same_columns=True
Sialkot         rows= 41208  first=2022-01-06 00:00:00  last=2026-09-18 23:00:00  duplicates=0  gaps=0  same_columns=True
Amritsar        rows= 41184  first=2022-01-08 00:00:00  last=2026-09-19 23:00:00  duplicates=0  gaps=0  sa

**What this cell does:** This cell loads the 11 air quality files (7 from Pakistan and 4 from Indian Punjab) into one table. It skips the 3 metadata lines at the top of each file, gives the pollutant columns short names, and adds `city` and `country` columns so every row shows where it came from. The files are stacked city by city, with Pakistan first and then India. The checks on each file found no duplicate timestamps, no gaps in the hourly timeline, and the same columns in every file. The raw time is in GMT, so 5 hours were added to get local time, which was then split into a `date` and an `hour` column. Finally, the data was cut to 2022-2023, which reduced it from 453,192 rows to 191,153 rows. 2024 is kept out for Week 14, and 2025-2026 is outside the project scope.

In [ ]:
# Save the stacked table as a CSV file so you can open it in Excel
# (it goes in the Dataset folder, which Git ignores, so it won't be uploaded)



# aq.to_csv("Dataset/aq_stacked_check.csv", index=False)
# print("Saved:", len(aq), "rows")

Saved: 191153 rows


In [6]:
# ============================================================
# Cell 2: STEP 4 - Missing data report
# ============================================================

# The 7 pollutant columns (we will use this list many times)
pollutants = ["pm10_ugm3", "pm2_5_ugm3", "no2_ugm3", "so2_ugm3",
              "o3_ugm3", "co_ugm3", "dust_ugm3"]

# ------------------------------------------------------------
# Table 1: missing count and percent for EVERY column
# ------------------------------------------------------------
table_all = pd.DataFrame({
    "missing_count": aq.isna().sum(),                    # how many empty cells
    "missing_percent": (aq.isna().mean() * 100).round(2) # same, as a percent
})
print("=== Table 1: missing values in every column ===")
print(table_all)

# ------------------------------------------------------------
# Table 2: missing percent for each city and each pollutant
# ------------------------------------------------------------
# isna() gives True/False for each cell, and mean() of True/False is the share of Trues
table_city = aq[pollutants].isna().groupby(aq["city"], sort=False).mean() * 100
print("\n=== Table 2: percent missing per city and pollutant ===")
print(table_city.round(2))

# ------------------------------------------------------------
# Check: do all pollutants go missing together?
# ------------------------------------------------------------
rows_all_missing = aq[pollutants].isna().all(axis=1).sum()   # every pollutant empty
rows_any_missing = aq[pollutants].isna().any(axis=1).sum()   # at least one empty
print("\nRows where ALL pollutants are missing:", rows_all_missing)
print("Rows where ANY pollutant is missing   :", rows_any_missing)
# If the two numbers are equal, pollutants are always missing together

# ------------------------------------------------------------
# Table 3: first time with real data, per city
# ------------------------------------------------------------
# Keep only rows where pm2_5 has a value, then find the earliest time for each city
rows_with_data = aq[aq["pm2_5_ugm3"].notna()]
first_valid = rows_with_data.groupby("city", sort=False)["time_local"].min()
print("\n=== Table 3: first hour with data, per city ===")
print(first_valid)

# ------------------------------------------------------------
# Table 4: longest run of missing hours in a row, per city
# ------------------------------------------------------------
print("\n=== Table 4: longest run of missing hours, per city ===")
for city in aq["city"].unique():
    # Take this city's pm2_5 column and mark the missing hours as True/False
    is_missing = aq[aq["city"] == city]["pm2_5_ugm3"].isna()

    current_run = 0     # how many missing hours in a row right now
    longest_run = 0     # the longest run we have seen so far
    for value in is_missing:
        if value:                      # this hour is missing
            current_run = current_run + 1
            if current_run > longest_run:
                longest_run = current_run
        else:                          # this hour has data, so the run is broken
            current_run = 0

    print(f"{city:15s} longest run = {longest_run} hours (= {longest_run / 24:.0f} days)")

# ------------------------------------------------------------
# Table 5: percent of hours missing, month by month
# ------------------------------------------------------------
month_labels = aq["date"].dt.to_period("M").astype(str)     # for example "2022-08"
missing_by_month = aq["pm2_5_ugm3"].isna().groupby(month_labels).mean() * 100
print("\n=== Table 5: percent missing per month (all cities together) ===")
print(missing_by_month.round(1))

=== Table 1: missing values in every column ===
            missing_count  missing_percent
city                    0              0.0
country                 0              0.0
time_utc                0              0.0
time_local              0              0.0
date                    0              0.0
hour                    0              0.0
pm10_ugm3           55248             28.9
pm2_5_ugm3          55248             28.9
no2_ugm3            55248             28.9
so2_ugm3            55248             28.9
o3_ugm3             55248             28.9
co_ugm3             55248             28.9
dust_ugm3           55248             28.9

=== Table 2: percent missing per city and pollutant ===
                pm10_ugm3  pm2_5_ugm3  no2_ugm3  so2_ugm3  o3_ugm3  co_ugm3  \
city                                                                          
Faisalabad          28.97       28.97     28.97     28.97    28.97    28.97   
Islamabad           28.97       28.97     28.97     28.9

**What this cell does and what we found:** This cell counts the missing values in every column, for each city and each pollutant, and also finds when the data starts, the longest run of missing hours, and the missing share per month. About 28.9% of the pollutant values are missing, and the amount is almost the same for every city (28.97% for Pakistan and 28.78% for India). All 7 pollutants are missing in exactly the same rows, so a row either has all its values or none. All of the missing values are in one continuous block at the start, from January 2022 until 2022-08-04 at 05:00 local time (5,040 hours, or 210 days, for Pakistan and 4,992 hours, or 208 days, for India). From that moment on there are no missing values at all. This means the data is not randomly missing: the source simply has no values before August 2022. So we will not fill these rows in, because that would mean inventing 7 months of pollution. We will remove them in the next step and work with data from August 2022 onwards.
The small difference between Pakistan (28.97%) and India (28.78%) is only because the Pakistan files were downloaded starting two days earlier, while the real data starts at the same time in all cities.

In [7]:
# ============================================================
# Cell 3: STEP 5 - Missing data strategy and cleaning
# ============================================================

# ------------------------------------------------------------
# Decision A: negative values are impossible for a concentration.
# First count them, then set them to 0.
# (We do this because the log transform later can't handle them)
# ------------------------------------------------------------
print("=== Negative values before cleaning ===")
for col in pollutants:
    print(col, ":", (aq[col] < 0).sum())

aq_clean = aq.copy()                                   # work on a copy, keep aq safe
aq_clean[pollutants] = aq_clean[pollutants].clip(lower=0)   # anything below 0 becomes 0

# ------------------------------------------------------------
# Decision B: drop the rows where ALL pollutants are empty.
# These are Jan to Aug 4, 2022, when the data source simply has no values.
# We do NOT fill them, because inventing 7 months of pollution would be wrong.
# ------------------------------------------------------------
rows_before = len(aq_clean)
aq_clean = aq_clean.dropna(subset=pollutants, how="all").reset_index(drop=True)
print(f"\nRows before dropping empty period: {rows_before}")
print(f"Rows after dropping empty period : {len(aq_clean)}")

# ------------------------------------------------------------
# Decision C: fill SHORT gaps only (6 hours or fewer) by drawing a straight
# line between the hour before and the hour after the gap.
# Long gaps are left empty.
# ------------------------------------------------------------
missing_before_fill = aq_clean[pollutants].isna().sum().sum()   # total empty cells now

for city in aq_clean["city"].unique():
    this_city = aq_clean["city"] == city          # True for this city's rows only
    aq_clean.loc[this_city, pollutants] = aq_clean.loc[this_city, pollutants].interpolate(
        limit=6,                # fill at most 6 hours in a row
        limit_area="inside"     # only fill between two known values, never at the ends
    )

missing_after_fill = aq_clean[pollutants].isna().sum().sum()
print(f"\nEmpty cells before filling: {missing_before_fill}")
print(f"Empty cells after filling : {missing_after_fill}")

# ------------------------------------------------------------
# Final check of the cleaned hourly table
# ------------------------------------------------------------
print("\nMissing cells per pollutant after cleaning:")
print(aq_clean[pollutants].isna().sum())
print("\nDate range:", aq_clean["date"].min(), "to", aq_clean["date"].max())
print("Shape:", aq_clean.shape)

=== Negative values before cleaning ===
pm10_ugm3 : 0
pm2_5_ugm3 : 0
no2_ugm3 : 19
so2_ugm3 : 0
o3_ugm3 : 197
co_ugm3 : 0
dust_ugm3 : 2

Rows before dropping empty period: 191153
Rows after dropping empty period : 135905

Empty cells before filling: 0
Empty cells after filling : 0

Missing cells per pollutant after cleaning:
pm10_ugm3     0
pm2_5_ugm3    0
no2_ugm3      0
so2_ugm3      0
o3_ugm3       0
co_ugm3       0
dust_ugm3     0
dtype: int64

Date range: 2022-08-04 00:00:00 to 2023-12-31 00:00:00
Shape: (135905, 13)


**What this cell does and what we found:** This cell cleans the hourly air quality table with three decisions. First, it counts negative pollutant values and sets them to 0, because a concentration cannot be negative and the log transform used later cannot handle them. There were only 218 such cells (19 for NO2, 197 for O3 and 2 for dust), out of about 950,000 values, so this change is tiny. Second, it removes every row where all pollutants are empty. These are the rows from January 2022 until 4 August 2022, when the data source has no values, so we do not fill them in, because that would mean inventing 7 months of pollution. This removed 55,248 rows and left 135,905 (12,355 hourly rows for each of the 11 cities). The cost is that we lose the first 7 months of 2022 Third, it fills short gaps of up to 6 hours with a straight line between the known values before and after the gap; (linear interpolation (with a gap limit of 6 hours), and it counts as a form of time-series imputation). After the empty period is removed there are no empty cells left at all, so nothing needed to be filled, and the table is fully complete from 4 August 2022 to 31 December 2023. The listwise deletion of the early months is the main possible source of bias, because the remaining data covers fewer winters than we wanted.

In [ ]:
# Save the cleaned hourly table (after cell 3) so we can look at it in Excel


# aq_clean.to_csv("Dataset/aq_hourly_clean.csv", index=False)
# print("Saved:", len(aq_clean), "rows")

Saved: 135905 rows


In [9]:
# ============================================================
# Cell 4: STEP 6 - Turn hourly data into one row per city per day
# ============================================================

# Pollutants where we also want the daily MAXIMUM (dust: mean only)
max_pollutants = ["pm10_ugm3", "pm2_5_ugm3", "no2_ugm3",
                  "so2_ugm3", "o3_ugm3", "co_ugm3"]

# Group the hourly rows so that each (city, date) is one group
# sort=False keeps our city order (Pakistan first, then India)
groups = aq_clean.groupby(["city", "date"], sort=False)

# Daily average of all 7 pollutants, daily maximum of 6 pollutants
daily_mean = groups[pollutants].mean()
daily_max = groups[max_pollutants].max()

# How many hours with real data each day has (should be 24 for a full day)
valid_hours = groups["pm2_5_ugm3"].count()

# Rename columns, for example pm10_ugm3 -> pm10_mean_ugm3 and pm10_max_ugm3
daily_mean.columns = [c.replace("_ugm3", "_mean_ugm3") for c in daily_mean.columns]
daily_max.columns = [c.replace("_ugm3", "_max_ugm3") for c in daily_max.columns]

# Put the three pieces side by side into one table
daily = pd.concat([daily_mean, daily_max], axis=1)
daily["valid_hours"] = valid_hours
daily = daily.reset_index()        # city and date become normal columns again

# Add the country column back (look it up from the hourly table)
country_of_city = aq_clean.drop_duplicates("city").set_index("city")["country"]
daily["country"] = daily["city"].map(country_of_city)

# ------------------------------------------------------------
# Rule: keep a day only if it has at least 18 of 24 hours of data
# ------------------------------------------------------------
days_before = len(daily)
daily = daily[daily["valid_hours"] >= 18].reset_index(drop=True)
print(f"Days before the 18-hour rule: {days_before}")
print(f"Days after the 18-hour rule : {len(daily)}")

# Put the columns in a clean order: identifiers first
first_columns = ["city", "country", "date", "valid_hours"]
other_columns = [c for c in daily.columns if c not in first_columns]
daily = daily[first_columns + other_columns]

# ------------------------------------------------------------
# Checks
# ------------------------------------------------------------
print("\nShape:", daily.shape)
print("Duplicate city+date rows:", daily.duplicated(subset=["city", "date"]).sum())
print("\nDays per city:")
print(daily.groupby("city", sort=False)["date"].agg(["min", "max", "count"]))
print("\nHours of data per day (valid_hours):")
print(daily["valid_hours"].value_counts())
print("\nFirst 5 rows:")
print(daily.head())

Days before the 18-hour rule: 5665
Days after the 18-hour rule : 5665

Shape: (5665, 17)
Duplicate city+date rows: 0

Days per city:
                      min        max  count
city                                       
Faisalabad     2022-08-04 2023-12-31    515
Islamabad      2022-08-04 2023-12-31    515
Lahore         2022-08-04 2023-12-31    515
Multan         2022-08-04 2023-12-31    515
Rahim Yar Khan 2022-08-04 2023-12-31    515
Rawalpindi     2022-08-04 2023-12-31    515
Sialkot        2022-08-04 2023-12-31    515
Amritsar       2022-08-04 2023-12-31    515
Ferozepur      2022-08-04 2023-12-31    515
Jalandhar      2022-08-04 2023-12-31    515
Ludhiana       2022-08-04 2023-12-31    515

Hours of data per day (valid_hours):
valid_hours
24    5654
19      11
Name: count, dtype: int64

First 5 rows:
         city country       date  valid_hours  pm10_mean_ugm3  \
0  Faisalabad      PK 2022-08-04           19       79.826316   
1  Faisalabad      PK 2022-08-05           24       

**What this cell does and what we found:** This cell turns the hourly table into a daily table, where one row is one city on one day. For each day it calculates the average of all 7 pollutants and the highest hourly value of 6 of them (dust gets an average only), and counts how many hours of real data the day has (`valid_hours`). We keep a day only if it has at least 18 of 24 hours, so a day with very little data cannot give a misleading average. The result has 5,665 rows (11 cities × 515 days, from 4 August 2022 to 31 December 2023) and 17 columns, with no duplicate city and date combinations. 5,654 days have all 24 hours. The other 11 days have 19 hours, which is the first day (4 August 2022) in each city, because the data starts at 05:00 local time. These days still pass the 18-hour rule, so no day was removed, but their averages use fewer hours than the others. The daily maximum is kept next to the daily average because it shows short spikes that the average hides. For example, on 8 August 2022 Faisalabad's NO2 average was 32.7 but its highest hour reached 130.5. This daily table is the dataset we will transform and use to build the smog index.

In [10]:
# Save the daily table (after cell 4) so we can look at it in Excel
daily.to_csv("Dataset/aq_daily.csv", index=False)
print("Saved:", len(daily), "rows")

Saved: 5665 rows
